In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, col, rand, when, expr
import time

In [26]:
spark = SparkSession.builder \
    .appName("Compendious Spark Optimization Lab") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

In [27]:
spark.sparkContext.master

'local[*]'

In [28]:
# Small dimension
countries = [
    ("KE","Kenya"),("UG","Uganda"),("TZ","Tanzania"),
    ("NG","Nigeria"),("SA","South Africa")
]
countries_df = spark.createDataFrame(countries, ["country_code","country_name"])
countries_df.show()

+------------+------------+
|country_code|country_name|
+------------+------------+
|          KE|       Kenya|
|          UG|      Uganda|
|          TZ|    Tanzania|
|          NG|     Nigeria|
|          SA|South Africa|
+------------+------------+



In [29]:
# Large fact
'''
transactions_df = (
    spark.range(0, 8_000_000)
    .withColumnRenamed("id", "txn_id")
    .withColumn(
        "country_code",
        when(col("txn_id") % 5 == 0, "KE")
        .when(col("txn_id") % 5 == 1, "UG")
        .when(col("txn_id") % 5 == 2, "TZ")
        .when(col("txn_id") % 5 == 3, "NG")
        .otherwise("SA")
    )
)
'''
transactions_df = (
    spark.range(0, 8_000_000)
    .withColumnRenamed("id", "txn_id")
    .withColumn(
        "country_code",
        expr("""
            CASE
                WHEN txn_id % 5 = 0 THEN 'KE'
                WHEN txn_id % 5 = 1 THEN 'UG'
                WHEN txn_id % 5 = 2 THEN 'TZ'
                WHEN txn_id % 5 = 3 THEN 'NG'
                ELSE 'SA'
            END
        """)
    )
)
transactions_df.show()

+------+------------+
|txn_id|country_code|
+------+------------+
|     0|          KE|
|     1|          UG|
|     2|          TZ|
|     3|          NG|
|     4|          SA|
|     5|          KE|
|     6|          UG|
|     7|          TZ|
|     8|          NG|
|     9|          SA|
|    10|          KE|
|    11|          UG|
|    12|          TZ|
|    13|          NG|
|    14|          SA|
|    15|          KE|
|    16|          UG|
|    17|          TZ|
|    18|          NG|
|    19|          SA|
+------+------------+
only showing top 20 rows



In [30]:
join_df = transactions_df.join(
    countries_df,
    on="country_code",
    how="inner"
)

join_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [country_code])
:- Project [txn_id#213L, CASE WHEN ((txn_id#213L % cast(5 as bigint)) = cast(0 as bigint)) THEN KE WHEN ((txn_id#213L % cast(5 as bigint)) = cast(1 as bigint)) THEN UG WHEN ((txn_id#213L % cast(5 as bigint)) = cast(2 as bigint)) THEN TZ WHEN ((txn_id#213L % cast(5 as bigint)) = cast(3 as bigint)) THEN NG ELSE SA END AS country_code#215]
:  +- Project [id#211L AS txn_id#213L]
:     +- Range (0, 8000000, step=1, splits=Some(6))
+- LogicalRDD [country_code#198, country_name#199], false

== Analyzed Logical Plan ==
country_code: string, txn_id: bigint, country_name: string
Project [country_code#215, txn_id#213L, country_name#199]
+- Join Inner, (country_code#215 = country_code#198)
   :- Project [txn_id#213L, CASE WHEN ((txn_id#213L % cast(5 as bigint)) = cast(0 as bigint)) THEN KE WHEN ((txn_id#213L % cast(5 as bigint)) = cast(1 as bigint)) THEN UG WHEN ((txn_id#213L % cast(5 as bigint)) = cast(2 as bigint)) THEN TZ WHEN ((t

In [31]:
# Broadcast join
broadcast_join_df = transactions_df.join(broadcast(countries_df),"country_code")
broadcast_join_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [country_code])
:- Project [txn_id#213L, CASE WHEN ((txn_id#213L % cast(5 as bigint)) = cast(0 as bigint)) THEN KE WHEN ((txn_id#213L % cast(5 as bigint)) = cast(1 as bigint)) THEN UG WHEN ((txn_id#213L % cast(5 as bigint)) = cast(2 as bigint)) THEN TZ WHEN ((txn_id#213L % cast(5 as bigint)) = cast(3 as bigint)) THEN NG ELSE SA END AS country_code#215]
:  +- Project [id#211L AS txn_id#213L]
:     +- Range (0, 8000000, step=1, splits=Some(6))
+- ResolvedHint (strategy=broadcast)
   +- LogicalRDD [country_code#198, country_name#199], false

== Analyzed Logical Plan ==
country_code: string, txn_id: bigint, country_name: string
Project [country_code#215, txn_id#213L, country_name#199]
+- Join Inner, (country_code#215 = country_code#198)
   :- Project [txn_id#213L, CASE WHEN ((txn_id#213L % cast(5 as bigint)) = cast(0 as bigint)) THEN KE WHEN ((txn_id#213L % cast(5 as bigint)) = cast(1 as bigint)) THEN UG WHEN ((txn_id#213L % cast(5 as bigint

In [32]:
join_df.count()

8000000

In [33]:
broadcast_join_df.count()

8000000

In [34]:
# Benchmark
def bench(df,label):
    s=time.time(); df.count()
    print(label, time.time()-s)

bench(transactions_df.join(countries_df,"country_code"),"Shuffle")
bench(transactions_df.join(broadcast(countries_df),"country_code"),"Broadcast")

Shuffle 2.9840099811553955
Broadcast 0.48851943016052246


In [35]:
join_df.sort('txn_id').show(10)

+------------+------+------------+
|country_code|txn_id|country_name|
+------------+------+------------+
|          KE|     0|       Kenya|
|          UG|     1|      Uganda|
|          TZ|     2|    Tanzania|
|          NG|     3|     Nigeria|
|          SA|     4|South Africa|
|          KE|     5|       Kenya|
|          UG|     6|      Uganda|
|          TZ|     7|    Tanzania|
|          NG|     8|     Nigeria|
|          SA|     9|South Africa|
+------------+------+------------+
only showing top 10 rows



In [36]:
broadcast_join_df.sort('txn_id').show()

+------------+------+------------+
|country_code|txn_id|country_name|
+------------+------+------------+
|          KE|     0|       Kenya|
|          UG|     1|      Uganda|
|          TZ|     2|    Tanzania|
|          NG|     3|     Nigeria|
|          SA|     4|South Africa|
|          KE|     5|       Kenya|
|          UG|     6|      Uganda|
|          TZ|     7|    Tanzania|
|          NG|     8|     Nigeria|
|          SA|     9|South Africa|
|          KE|    10|       Kenya|
|          UG|    11|      Uganda|
|          TZ|    12|    Tanzania|
|          NG|    13|     Nigeria|
|          SA|    14|South Africa|
|          KE|    15|       Kenya|
|          UG|    16|      Uganda|
|          TZ|    17|    Tanzania|
|          NG|    18|     Nigeria|
|          SA|    19|South Africa|
+------------+------+------------+
only showing top 20 rows

